# モデル評価・チューニング

モデルをより正確に評価し、最適なパラメータを探す手法を学びます。

| ステップ | 内容 |
|----------|------|
| 1 | 交差検証（Cross Validation） |
| 2 | グリッドサーチ（Grid Search） |
| 3 | 複数モデルの比較 |
| 4 | 学習曲線の可視化 |

## 0. データ準備

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X = iris.data
y = iris.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("データ準備完了:", X_scaled.shape)

## 1. 交差検証（Cross Validation）

データを k 分割して、k 回学習・評価を繰り返します。  
1回のホールドアウトより信頼性の高い評価ができます。

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

model = KNeighborsClassifier(n_neighbors=3)

# 5分割交差検証
scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')

print("各フォールドの正解率:", [f"{s:.4f}" for s in scores])
print(f"平均: {scores.mean():.4f} ± {scores.std():.4f}")

## 2. グリッドサーチ（Grid Search）

ハイパーパラメータの組み合わせを総当たりで試し、最良のものを自動で見つけます。

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_scaled, y)

print("最良パラメータ:", grid_search.best_params_)
print("最良スコア   :", f"{grid_search.best_score_:.4f}")

## 3. 複数モデルの比較

同じデータで複数のアルゴリズムを横並びで比較します。

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

models = {
    'KNN':              KNeighborsClassifier(n_neighbors=3),
    'Logistic Reg':     LogisticRegression(max_iter=200),
    'SVM':              SVC(),
    'Decision Tree':    DecisionTreeClassifier(random_state=42),
    'Random Forest':    RandomForestClassifier(random_state=42),
    'Gradient Boost':   GradientBoostingClassifier(random_state=42)
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
    results[name] = scores
    print(f"{name:18s}: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# 箱ひげ図で比較
plt.figure(figsize=(10, 5))
plt.boxplot(results.values(), labels=results.keys(), patch_artist=True)
plt.ylabel("正解率")
plt.title("モデル比較（5分割交差検証）")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 4. 学習曲線の可視化

学習データ数を変えながら精度を観察します。  
**過学習（overfitting）や学習不足（underfitting）の診断**に使います。

In [ ]:
from sklearn.model_selection import learning_curve

model = RandomForestClassifier(random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    model, X_scaled, y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, label='学習スコア', color='steelblue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='steelblue')
plt.plot(train_sizes, val_mean, label='検証スコア', color='orange')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color='orange')
plt.xlabel("学習データ数")
plt.ylabel("正解率")
plt.title("学習曲線（Random Forest）")
plt.legend()
plt.tight_layout()
plt.show()